In [2]:
import numpy as np
from get_court_position_methods import calibrate_camera, get_3d_position
import cv2
import numpy as np
import torch as t
import torchvision
import torchaudio
import albumentations as A
from ultralytics import YOLO
import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d
import filterpy
from filterpy.kalman import UnscentedKalmanFilter, IMMEstimator, MerweScaledSigmaPoints
from filterpy.common import Q_discrete_white_noise
from scipy.linalg import block_diag

In [7]:
def fx_ballistic(x, dt):
    """Балістична нелінійна модель: x_next = F*x * gravity_effect
    F-матриця для [x, vx, y, vy, z, vz]"""
    vx, vy, vz = x[1], x[3], x[5]
    F = np.array([[1, dt, 0, 0,  0,  0],                  # x
                  [0, 1,  0, 0,  0,  0],                  # vx
                  [0, 0,  1, dt, 0,  0],                  # y
                  [0, 0,  0, 1,  0,  0],                  # vy
                  [0, 0,  0, 0,  1,  dt],                 # z
                  [0, 0,  0, 0,  0,  1]], dtype=float)    # vz

    B = np.diag([0.5 * dt**2, dt, 0.5 * dt**2, dt, 0.5 * dt**2, dt])
    k = 0.0314
    v_mag = np.sqrt(x[1]**2 + x[3]**2 + x[5]**2)
    a_dx = -k * v_mag * vx
    a_dy = -k * v_mag * vy
    a_dz = -k * v_mag * vz
    u = np.array([a_dx, a_dx, a_dy - 9.81, a_dy - 9.81, a_dz, a_dz])

    x_next = np.dot(F, x) + np.dot(B, u)

    return x_next

def fx_hit(x, dt):
    """
    Модель удару: Constant Velocity.
    Стан: [x, vx, y, vy, z, vz]
    """
    F = np.array([[1, dt, 0, 0,  0,  0],
                  [0, 1,  0, 0,  0,  0],
                  [0, 0,  1, dt, 0,  0],
                  [0, 0,  0, 1,  0,  0],
                  [0, 0,  0, 0,  1,  dt],
                  [0, 0,  0, 0,  0,  1]], dtype=float)

    return np.dot(F, x)

def fx_bounce(x, dt):
    """Модель відскоку, марковська матриця"""
    espilon = 0.75 # Коефіцієнт реституції
    friction = 0.85 # Коефіцієнт тертя
    x_next = np.copy(x)
    vx_new = x[1] * friction
    vy_new = -x[3] * epsilon
    vz_new = x[5] * friction

    x_next[0] += vx_new * dt
    x_next[2] += xy_new * dt
    x_next[4] += vz_new * dt

    x_next[1] = vx_new
    x_next[3] = vy_new
    x_next[5] = vz_new

    return x_next

def get_dynamic_transition_matrix(y, vy):
    """
    Повертає марковську матрицю 3x3 для моделей: [Ballistic, Hit, Bounce]
    """
    M = np.array([[0.95, 0.04, 0.01],
                  [0.60, 0.40, 0.00],
                  [0.90, 0.00, 0.10]])

    if y < 0.3 and vy < 0:
        M[0] = [0.10, 0.05, 0.85]
        M[1] = [0.10, 0.05, 0.85]

    return M

In [6]:
# Для балістичної моделі Q мінімальна
q_var_ballistic = 0.1
q_b = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_ballistic)
Q_ballistic = block_diag(q_b, q_b, q_b)

# Для моделі удару Q велика
q_var_hit = 100
q_h = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_hit)
Q_hit = block_diag(q_h, q_h, q_h)

# Для відскоку Q середня
q_var_bounce = 5
q_bnc = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_bounce)
Q_bounce = block_diag(q_bnc, q_bnc, q_bnc)